# Experiment 2B: JSON rating template comparison

This notebook compares three black-box/API-style structural scoring approaches using Qwen 2.5 1.5B:

1. **score then justification**
2. **justification then score**
3. **no justification**

The model is asked to output JSON with a `risk_score` from 0 to 100. This is different from the logit-based yes/no check: it is meant to simulate a deployment where we cannot inspect model internals.

No fallback or seeded outputs are used. Every row is scored from the input dataset and template file.

In [ ]:
# ============================================================
# Config
# ============================================================

PASSWORD_DATA_CSV = "experiment2_template_sensitivity_passwords.csv"
RATING_TEMPLATES_CSV = "experiment2b_rating_templates.csv"

OUT_DIR = "results_exp2b"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRUST_REMOTE_CODE = True

# Set to a small number for smoke tests, then None for the real run.
RUN_FIRST_N_PASSWORDS = None

# Generation settings. Greedy is intentional for repeatability.
DO_SAMPLE = False
TEMPERATURE = None
TOP_P = None

# If the model emits broken JSON, the parser will try to recover the first {...} block.
PRINT_BAD_JSON_EXAMPLES = 15

import os
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# ============================================================
# Install/import
# ============================================================

import os, re, json, time, math, ast, subprocess, sys
from pathlib import Path
from typing import Dict, Any, Optional, List

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "torch"])
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import matplotlib.pyplot as plt
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib.pyplot as plt

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())


In [ ]:
# ============================================================
# Load inputs
# ============================================================

password_path = Path(PASSWORD_DATA_CSV)
template_path = Path(RATING_TEMPLATES_CSV)

if not password_path.exists():
    raise FileNotFoundError(f"Missing {password_path}. Upload/copy it into the Colab working directory.")
if not template_path.exists():
    raise FileNotFoundError(f"Missing {template_path}. Upload/copy it into the Colab working directory.")

df_pw = pd.read_csv(password_path)
df_tpl = pd.read_csv(template_path)

required_pw = {"password_id", "password", "expected_structural_label", "family", "condition"}
missing_pw = required_pw - set(df_pw.columns)
if missing_pw:
    raise ValueError(f"Password dataset missing columns: {missing_pw}")

required_tpl = {"template_id", "approach", "enabled", "max_new_tokens", "user_prompt"}
missing_tpl = required_tpl - set(df_tpl.columns)
if missing_tpl:
    raise ValueError(f"Template file missing columns: {missing_tpl}")

df_tpl = df_tpl[df_tpl["enabled"].astype(str).str.lower().isin(["true", "1", "yes"])]

if RUN_FIRST_N_PASSWORDS is not None:
    df_pw = df_pw.head(RUN_FIRST_N_PASSWORDS).copy()

print("password rows:", len(df_pw))
print("rating templates:", len(df_tpl))
display(df_tpl[["template_id", "approach", "max_new_tokens", "notes"]])
print(df_pw["expected_structural_label"].value_counts(dropna=False))


In [ ]:
# ============================================================
# Load model
# ============================================================

t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
)
model.eval()

elapsed = time.perf_counter() - t0
print(f"Loaded {MODEL_ID} in {elapsed:.1f}s")
print("device:", next(model.parameters()).device)


In [ ]:
# ============================================================
# Prompting + JSON parsing helpers
# ============================================================

SYSTEM_PROMPT = (
    "You are a password-risk classifier. "
    "You must return only valid JSON. "
    "Do not include markdown, code fences, or extra text."
)

VALID_CATEGORIES = {
    "none", "sequence", "phrase", "cultural_association",
    "repetition", "personal_info_like", "other"
}

def build_prompt(user_prompt: str) -> str:
    # Use chat template when available. This keeps Qwen in instruction-following mode.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return SYSTEM_PROMPT + "\n\n" + user_prompt + "\n"


def extract_json_object(text: str) -> Optional[str]:
    # Remove common code fences if present.
    t = text.strip()
    t = re.sub(r"^```(?:json)?", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"```$", "", t).strip()

    # Greedy enough for one flat object.
    start = t.find("{")
    end = t.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    return t[start:end+1]


def parse_model_json(text: str) -> Dict[str, Any]:
    js = extract_json_object(text)
    if js is None:
        return {"parse_ok": False, "parse_error": "no_json_object"}

    try:
        obj = json.loads(js)
    except Exception as e1:
        # Small fallback for single quotes/trailing commas. Still records if recovery was needed.
        try:
            fixed = re.sub(r",\s*}", "}", js)
            obj = ast.literal_eval(fixed)
            if not isinstance(obj, dict):
                return {"parse_ok": False, "parse_error": "literal_not_dict"}
            obj["_recovered_with_literal_eval"] = True
        except Exception as e2:
            return {"parse_ok": False, "parse_error": f"json_error: {e1}"}

    raw_score = obj.get("risk_score", None)
    try:
        score = int(round(float(raw_score)))
    except Exception:
        # Try extracting a number from a string like "80/100".
        m = re.search(r"-?\d+(?:\.\d+)?", str(raw_score))
        if not m:
            return {"parse_ok": False, "parse_error": "missing_or_bad_risk_score", "raw_obj": obj}
        score = int(round(float(m.group(0))))

    score = max(0, min(100, score))

    category = str(obj.get("category", "other")).strip()
    if category not in VALID_CATEGORIES:
        category = "other"

    justification = str(obj.get("justification", "")).strip()

    return {
        "parse_ok": True,
        "risk_score": score,
        "category": category,
        "justification": justification,
        "raw_obj": obj,
    }


@torch.no_grad()
def generate_json_rating(password: str, template_row: pd.Series) -> Dict[str, Any]:
    user_prompt = str(template_row["user_prompt"]).format(password=password)
    prompt = build_prompt(user_prompt)

    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = enc["input_ids"].shape[1]

    gen_kwargs = dict(
        max_new_tokens=int(template_row.get("max_new_tokens", 120)),
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.eos_token_id,
    )
    if TEMPERATURE is not None:
        gen_kwargs["temperature"] = TEMPERATURE
    if TOP_P is not None:
        gen_kwargs["top_p"] = TOP_P

    t0 = time.perf_counter()
    out = model.generate(**enc, **gen_kwargs)
    elapsed = time.perf_counter() - t0

    new_tokens = out[0, input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    parsed = parse_model_json(decoded)

    return {
        "raw_generation": decoded,
        "generation_seconds": elapsed,
        "num_prompt_tokens": input_len,
        "num_new_tokens": int(new_tokens.shape[0]),
        **parsed,
    }


In [ ]:
# ============================================================
# Run experiment
# ============================================================

rows = []
total = len(df_pw) * len(df_tpl)

for _, tpl in tqdm(df_tpl.iterrows(), total=len(df_tpl), desc="Templates"):
    for _, pw in tqdm(df_pw.iterrows(), total=len(df_pw), desc=str(tpl["template_id"]), leave=False):
        res = generate_json_rating(str(pw["password"]), tpl)
        rows.append({
            "password_id": pw["password_id"],
            "family": pw.get("family", ""),
            "condition": pw.get("condition", ""),
            "group_id": pw.get("group_id", ""),
            "password": pw["password"],
            "expected_structural_label": pw.get("expected_structural_label", ""),
            "structure_type": pw.get("structure_type", ""),
            "template_id": tpl["template_id"],
            "approach": tpl["approach"],
            "template_notes": tpl.get("notes", ""),
            **res,
        })

df_raw = pd.DataFrame(rows)
raw_path = Path(OUT_DIR) / "exp2b_json_rating_raw.csv"
df_raw.to_csv(raw_path, index=False)
print("saved", raw_path)
print("parse rate", df_raw["parse_ok"].mean())

display(df_raw.head())

bad = df_raw[~df_raw["parse_ok"]].head(PRINT_BAD_JSON_EXAMPLES)
if len(bad):
    print("Bad JSON examples:")
    display(bad[["template_id", "password_id", "password", "parse_error", "raw_generation"]])


In [ ]:
# ============================================================
# Metrics
# ============================================================

valid = df_raw[df_raw["parse_ok"]].copy()
valid["is_yes"] = valid["expected_structural_label"].eq("yes")
valid["is_no"] = valid["expected_structural_label"].eq("no")
valid["is_uncertain"] = valid["expected_structural_label"].eq("uncertain")


def auc_pairwise(scores_pos, scores_neg):
    pos = np.asarray(scores_pos, dtype=float)
    neg = np.asarray(scores_neg, dtype=float)
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    # P(score_pos > score_neg) + 0.5 P(tie)
    wins = 0.0
    total = 0
    for p in pos:
        wins += np.sum(p > neg) + 0.5 * np.sum(p == neg)
        total += len(neg)
    return wins / total

summary_rows = []
thresholds = [50, 60, 70, 80, 90]

for (template_id, approach), g in valid.groupby(["template_id", "approach"]):
    yes_scores = g.loc[g["is_yes"], "risk_score"]
    no_scores = g.loc[g["is_no"], "risk_score"]
    uncertain_scores = g.loc[g["is_uncertain"], "risk_score"]

    row = {
        "template_id": template_id,
        "approach": approach,
        "n_valid": len(g),
        "parse_rate": len(g) / len(df_pw),
        "mean_score_yes": yes_scores.mean(),
        "mean_score_no": no_scores.mean(),
        "mean_score_uncertain": uncertain_scores.mean(),
        "separation_yes_minus_no": yes_scores.mean() - no_scores.mean(),
        "auc_yes_vs_no": auc_pairwise(yes_scores, no_scores),
        "mean_generation_seconds": g["generation_seconds"].mean(),
        "mean_new_tokens": g["num_new_tokens"].mean(),
    }
    for th in thresholds:
        row[f"tpr_yes_at_{th}"] = (yes_scores >= th).mean() if len(yes_scores) else np.nan
        row[f"fpr_no_at_{th}"] = (no_scores >= th).mean() if len(no_scores) else np.nan
        row[f"uncertain_flag_rate_at_{th}"] = (uncertain_scores >= th).mean() if len(uncertain_scores) else np.nan
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values(
    ["auc_yes_vs_no", "separation_yes_minus_no", "parse_rate"],
    ascending=[False, False, False]
).reset_index(drop=True)

summary_path = Path(OUT_DIR) / "exp2b_json_rating_template_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved", summary_path)
display(summary)

approach_summary = summary.groupby("approach").agg(
    n_templates=("template_id", "count"),
    mean_auc=("auc_yes_vs_no", "mean"),
    best_auc=("auc_yes_vs_no", "max"),
    mean_sep=("separation_yes_minus_no", "mean"),
    best_sep=("separation_yes_minus_no", "max"),
    mean_parse_rate=("parse_rate", "mean"),
    mean_generation_seconds=("mean_generation_seconds", "mean"),
).reset_index().sort_values("best_auc", ascending=False)

approach_path = Path(OUT_DIR) / "exp2b_json_rating_approach_summary.csv"
approach_summary.to_csv(approach_path, index=False)
print("saved", approach_path)
display(approach_summary)


In [ ]:
# ============================================================
# Label/category summaries
# ============================================================

label_summary = valid.groupby(["template_id", "approach", "expected_structural_label"]).agg(
    n=("password_id", "count"),
    mean_score=("risk_score", "mean"),
    median_score=("risk_score", "median"),
    std_score=("risk_score", "std"),
).reset_index()

label_path = Path(OUT_DIR) / "exp2b_json_rating_label_summary.csv"
label_summary.to_csv(label_path, index=False)
print("saved", label_path)
display(label_summary.head(30))

category_summary = valid.groupby(["template_id", "approach", "category", "expected_structural_label"]).size().reset_index(name="n")
category_path = Path(OUT_DIR) / "exp2b_json_rating_category_counts.csv"
category_summary.to_csv(category_path, index=False)
print("saved", category_path)


In [ ]:
# ============================================================
# Figures
# ============================================================

fig_dir = Path(OUT_DIR) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Template AUC/separation scatter
plt.figure(figsize=(9, 6))
for approach, g in summary.groupby("approach"):
    plt.scatter(g["separation_yes_minus_no"], g["auc_yes_vs_no"], label=approach)
    for _, r in g.iterrows():
        plt.text(r["separation_yes_minus_no"], r["auc_yes_vs_no"], r["template_id"], fontsize=7)
plt.xlabel("Mean score gap: yes - no")
plt.ylabel("AUC: yes vs no")
plt.title("Experiment 2B JSON rating templates")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = fig_dir / "exp2b_template_auc_vs_gap.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved", path)

# 2. Approach summary bar chart
plt.figure(figsize=(8, 5))
approach_summary_sorted = approach_summary.sort_values("best_auc", ascending=False)
plt.bar(approach_summary_sorted["approach"], approach_summary_sorted["best_auc"])
plt.ylabel("Best AUC among templates")
plt.title("Best structural separation by JSON-output approach")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
path = fig_dir / "exp2b_approach_best_auc.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved", path)

# 3. Best template score distribution
best_template = summary.iloc[0]["template_id"]
best = valid[valid["template_id"].eq(best_template)]
plt.figure(figsize=(8, 5))
labels = ["no", "uncertain", "yes"]
data = [best.loc[best["expected_structural_label"].eq(label), "risk_score"].dropna() for label in labels]
plt.boxplot(data, labels=labels)
plt.ylabel("risk_score")
plt.title(f"Risk-score distribution for best template: {best_template}")
plt.tight_layout()
path = fig_dir / "exp2b_best_template_score_distribution.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved", path)


In [ ]:
# ============================================================
# Recommended template + examples
# ============================================================

best_row = summary.iloc[0].to_dict()
print("Recommended template by AUC then gap:")
print(json.dumps({k: best_row[k] for k in ["template_id", "approach", "auc_yes_vs_no", "separation_yes_minus_no", "parse_rate", "mean_generation_seconds"]}, indent=2))

best_template_id = best_row["template_id"]
best_examples = valid[valid["template_id"].eq(best_template_id)].copy()

print("\nHighest-scored no/random examples (possible false positives):")
display(best_examples[best_examples["expected_structural_label"].eq("no")].sort_values("risk_score", ascending=False).head(15)[
    ["password_id", "password", "family", "condition", "risk_score", "category", "justification", "raw_generation"]
])

print("\nLowest-scored yes/structured examples (possible false negatives):")
display(best_examples[best_examples["expected_structural_label"].eq("yes")].sort_values("risk_score", ascending=True).head(15)[
    ["password_id", "password", "family", "condition", "risk_score", "category", "justification", "raw_generation"]
])

rec = {
    "best_template_id": best_template_id,
    "best_approach": best_row["approach"],
    "auc_yes_vs_no": float(best_row["auc_yes_vs_no"]),
    "separation_yes_minus_no": float(best_row["separation_yes_minus_no"]),
    "parse_rate": float(best_row["parse_rate"]),
}
rec_path = Path(OUT_DIR) / "exp2b_recommended_json_rating_template.json"
with open(rec_path, "w") as f:
    json.dump(rec, f, indent=2)
print("saved", rec_path)


## Interpretation notes

Use this experiment to answer two questions:

1. Does a black-box JSON rating prompt separate structured passwords from random-looking controls better than yes/no logits did?
2. Does asking for a justification before the score help, hurt, or make parsing slower?

Do not treat the JSON `risk_score` as a calibrated probability. It is a model-produced rating. The useful metric is separation between expected-structured and expected-random rows, plus concrete false positives/false negatives.